<a href="https://colab.research.google.com/github/bekaeackonor/lab-4-llm-decision-support/blob/main/Kwasi_Ackonor_Lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# API-key setup
import os

# Google Colab (Secrets panel)
from google.colab import userdata
API_KEY = userdata.get("GROQ_API_KEY")

# OpenAI-compatible client (works for Groq and OpenAI)
from openai import OpenAI

client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.groq.com/openai/v1",
)
MODEL = "llama-3.3-70b-versatile"

print("Client ready.")

Client ready.


In [2]:
# Part 1.1 — Your first API call

def ask_llm(user_prompt, system_prompt="You are a helpful assistant.",
            temperature=0.7, max_tokens=500):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response, response.choices[0].message.content

# Call it once with a simple question
test_response, answer = ask_llm("What is microfinance, in one sentence?")

print("Answer:")
print(answer)
print("\nToken usage:")
print(test_response.usage)

Answer:
Microfinance refers to the provision of small-scale financial services, such as loans, savings, and insurance, to low-income individuals or groups who lack access to traditional banking services, often in developing countries.

Token usage:
CompletionUsage(completion_tokens=41, prompt_tokens=50, total_tokens=91, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.052112795, prompt_time=0.001693922, completion_time=0.135117405, total_time=0.136811327)


In [3]:
# Part 1.2 — Temperature: the randomness dial

question = "Suggest a name for a savings product for market traders in Accra."

print("=" * 60)
print("TEMPERATURE = 0.0")
print("=" * 60)
temp_0_answers = []
for i in range(5):
    _, answer = ask_llm(question, temperature=0.0, max_tokens=50)
    temp_0_answers.append(answer)
    print(f"\n[Run {i+1}]\n{answer}")

print("\n" + "=" * 60)
print("TEMPERATURE = 1.2")
print("=" * 60)
temp_12_answers = []
for i in range(5):
    _, answer = ask_llm(question, temperature=1.2, max_tokens=50)
    temp_12_answers.append(answer)
    print(f"\n[Run {i+1}]\n{answer}")

TEMPERATURE = 0.0

[Run 1]
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

[Run 2]
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

[Run 3]
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

[Run 4]
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name could resonate with market traders.
2. **

[Run 5]
Here are a few suggestions for a savings product for market traders in Accra:

1. **Makola Save**: "Makola" is a well-known market in Accra, so this name

In [4]:
LETTERS = {
"L001": """Dear Sir/Madam, My name is Akosua Mensah and I have been selling provisions at Makola Market for 12 years. I am applying for a loan of GHS 8,000 to buy a deep freezer and expand into frozen foods. My current stall makes about GHS 900 profit each month. I have saved GHS 2,500 with your susu scheme over the past two years and I have never missed a contribution. I can repay GHS 450 monthly over 20 months. My sister, a teacher, will stand as my guarantor. Thank you for considering my application.""",

"L002": """Hello, I am Kwame Boateng, a commercial driver in Kumasi. I need GHS 25,000 urgently to repair my trotro engine and settle some personal debts. Business has been slow but it will surely pick up after the festive season. I can pay back whenever the money comes. I do not have collateral at the moment but God willing everything will be fine. Please help me quickly.""",

"L003": """Dear Loan Committee, I am Efua Darko, owner of Darko Fashions, a registered dressmaking business in Takoradi (registration no. BN-2019-4482). I employ three apprentices. I request GHS 15,000 to purchase two industrial sewing machines and fabric stock ahead of the Christmas season. Last year my December revenue alone was GHS 22,000; monthly profit averages GHS 2,800. I hold a fixed deposit of GHS 5,000 with GCB which I can pledge. Proposed repayment: GHS 1,100 monthly for 15 months. Attached are my sales records for the past 18 months.""",

"L004": """Good day, My name is Yaw Owusu. I want a loan for my poultry farm at Nsawam. The amount is GHS 12,000 for feed and 500 new layers. I started the farm last year. Sometimes I make good money, around GHS 1,500 in a good month, but bird flu affected us in March and I lost many birds. I am rebuilding now. I can repay in 18 months. My uncle has agreed to guarantee the loan with his taxi.""",

"L005": """Dear Manager, I am writing on behalf of the Adenta Women's Weaving Cooperative (14 members). We seek GHS 30,000 to buy a bulk order of yarn directly from the factory, cutting out middlemen and raising our margins from 15% to about 35%. The cooperative has operated for 6 years and holds GHS 9,000 in our group account. We propose repayment of GHS 2,000 monthly over 16 months, backed by our group savings and joint liability agreement.""",

"L006": """Hi, This is Kofi. I saw your advert. I want GHS 50,000 to start a car washing business, a provision shop, and also import phones from Dubai. I am 22 and full of energy. I have not started any of these yet but my friends say I am very business minded. I will pay back in one year when the businesses are booming. No collateral but I am trustworthy.""",
}

GOLD = {
"L001": {"applicant_name": "Akosua Mensah", "amount_ghs": 8000,  "purpose": "buy deep freezer / expand into frozen foods",
         "monthly_profit_ghs": 900,  "has_collateral_or_guarantor": True,  "repayment_months": 20},
"L003": {"applicant_name": "Efua Darko",    "amount_ghs": 15000, "purpose": "industrial sewing machines and fabric stock",
         "monthly_profit_ghs": 2800, "has_collateral_or_guarantor": True,  "repayment_months": 15},
"L006": {"applicant_name": "Kofi",          "amount_ghs": 50000, "purpose": "car wash, provision shop, phone imports",
         "monthly_profit_ghs": None, "has_collateral_or_guarantor": False, "repayment_months": 12},
}

print(f"{len(LETTERS)} letters loaded.")

6 letters loaded.


In [5]:
# Part 3.1 — Component 1: Summarization

# --- V1: naive prompt ---
SUMMARY_PROMPT_V1 = "Summarize this:"

def summarize_v1(letter_text):
    _, answer = ask_llm(f"{SUMMARY_PROMPT_V1}\n\n{letter_text}", temperature=0.7, max_tokens=200)
    return answer

print("=" * 60)
print("V1 — L002")
print("=" * 60)
print(summarize_v1(LETTERS["L002"]))

print("\n" + "=" * 60)
print("V1 — L006")
print("=" * 60)
print(summarize_v1(LETTERS["L006"]))

V1 — L002
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's struggling due to slow business, but expects it to improve after the festive season. He doesn't have collateral and is relying on his future earnings to repay the loan.

V1 — L006
Kofi, a 22-year-old, is seeking GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business from Dubai. Although he has no experience and no collateral, he claims to be business-minded and promises to repay the loan within a year when his businesses are successful, relying on his personal trustworthiness.


In [6]:
# --- V2: structured system + user prompt, temperature=0 ---

SUMMARY_SYSTEM_PROMPT = """You are an assistant to a microfinance loan officer. \
Your job is to summarize loan application letters into short, factual briefs.

Rules:
- Write exactly 3-4 sentences.
- Be strictly factual and neutral. Do not add opinions, judgments, or embellishments.
- Do NOT invent, assume, or infer any detail that is not explicitly stated in the letter.
- If a detail (e.g. collateral, repayment plan) is missing, do not mention it — do not guess.
- Use plain, professional language a busy loan officer can scan in seconds."""

def summarize_v2(letter_text):
    user_prompt = f"Summarize this loan application:\n\n{letter_text}"
    _, answer = ask_llm(user_prompt, system_prompt=SUMMARY_SYSTEM_PROMPT,
                         temperature=0.0, max_tokens=200)
    return answer

print("=" * 60)
print("V2 — L002")
print("=" * 60)
print(summarize_v2(LETTERS["L002"]))

print("\n" + "=" * 60)
print("V2 — L006")
print("=" * 60)
print(summarize_v2(LETTERS["L006"]))

V2 — L002
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. The loan is needed to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to improve after the festive season. He does not have collateral at the moment and has not specified a repayment plan.

V2 — L006
Kofi is applying for a loan of GHS 50,000 to start three businesses: a car washing business, a provision shop, and a phone import business. He is 22 years old and has not yet started any of these ventures. Kofi plans to repay the loan in one year. He does not have collateral to offer, but claims to be trustworthy.


In [7]:
# --- Side-by-side comparison ---
for lid in ["L002", "L006"]:
    print("#" * 70)
    print(f"LETTER {lid}")
    print("#" * 70)
    print("\n--- V1 ---")
    print(summarize_v1(LETTERS[lid]))
    print("\n--- V2 ---")
    print(summarize_v2(LETTERS[lid]))
    print()

######################################################################
LETTER L002
######################################################################

--- V1 ---
Kwame Boateng, a commercial driver in Kumasi, is seeking a loan of GHS 25,000 to repair his vehicle's engine and pay off personal debts. He's experiencing a slow period in business but is optimistic it will improve after the festive season. He doesn't have collateral to offer but is asking for help and promising to repay the loan as soon as possible.

--- V2 ---
Kwame Boateng, a commercial driver in Kumasi, is applying for a loan of GHS 25,000. The loan is needed to repair his trotro engine and settle personal debts. He mentions that business has been slow, but expects it to improve after the festive season. He does not have collateral at the moment and has not specified a repayment plan.

######################################################################
LETTER L006
####################################################